# Capítulo 2 — Por dentro do modelo: tokens, amostragem e saídas estruturadas

## Missão
Entender, mexendo nos números, as três coisas que mais afetam o seu produto: **tokens** (custo e limite de contexto), **amostragem** (por que o modelo responde diferente a cada vez) e **saídas estruturadas** (como fazer o modelo devolver dados que o seu código consegue usar).

**No fim deste caderno você terá:** o módulo `meu_assistente/estruturado.py`, que transforma um chamado de cliente em um objeto validado (categoria, prioridade, resumo).

**Leitura essencial (opcional, livro em inglês):**
- Training Data: Multilingual Models · PDF p. 75
- Sampling Fundamentals e Sampling Strategies · PDF p. 112
- Test Time Compute · PDF p. 120
- Structured Outputs · PDF p. 123
- The Probabilistic Nature of AI · PDF p. 129

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap02_por_dentro_do_modelo_tokens_amostragem_e_said.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
O modelo não lê palavras: lê **tokens**, pedaços de texto. A cada passo ele calcula uma pontuação (*logit*) para cada token possível, transforma em probabilidades (*softmax*) e **sorteia** o próximo. A temperatura, o top-k e o top-p mudam esse sorteio. Por isso a mesma pergunta pode ter respostas diferentes, e é por isso também que vale a pena forçar um formato (JSON) quando o resultado vai para um programa.

### 2.1 **Tokens na prática.** Com `tiktoken` (`tiktoken.get_encoding("o200k_base")`), codifique a frase "A Nimbus organiza as tarefas da sua equipe.". Mostre o número de tokens e cada pedaço decodificado (`enc.decode([t])`).

### 2.2 **Português custa mais?** Compare o número de tokens de 3 frases em português e das mesmas frases em inglês (use `contar_tokens`). Mostre a razão PT/EN de cada uma e a média.

### 2.3 **Temperatura à mão.** Com os logits `[2.0, 1.0, 0.5, 0.1]` para os tokens `["sim", "não", "talvez", "nunca"]`, calcule o softmax com temperatura (`logits / T`) para `T = 0.2, 1 e 2`. Mostre numa tabela. O que acontece com o token mais provável quando T cai?

### 2.4 **Top-k e top-p.** Escreva `amostrar(logits, T=1.0, top_k=None, top_p=None, rng=None)` que: aplica a temperatura; se houver `top_k`, mantém só os k maiores; se houver `top_p`, mantém os mais prováveis até somar p; renormaliza e sorteia um índice. Sorteie 2.000 vezes com `top_k=2` e com `top_p=0.8` e conte quantas vezes saiu cada token.

### 2.5 **Logprobs de verdade.** Peça ao modelo para completar "A capital da Austrália é" com `logprobs=True` e `top_logprobs=3`, `max_tokens=5`. Para cada token gerado, mostre a probabilidade (`np.exp(logprob)`) e as alternativas que o modelo considerou.

### 2.6 **Temperatura no modelo.** Peça 3 amostras (`n=3`) de um slogan para a Nimbus com `temperature=0` e depois com `temperature=1.2`. Quantas respostas **diferentes** saíram em cada caso?

### 2.7 **Gastar mais tempo de teste para acertar mais (autoconsistência).** Faça a pergunta "Uma equipe de 7 pessoas no plano Equipe paga quanto por mês, a R$ 29 por pessoa? Responda só com o número." com `n=5` e `temperature=0.8`. Extraia o número de cada resposta com regex e escolha o mais votado (`Counter`). Se nenhum número aparecer, mostre `None`.

### 2.8 **JSON com esquema.** Peça a classificação do chamado "Fui cobrado duas vezes este mês" usando `response_format` do tipo `json_schema`, com os campos `categoria` (um de `CATEGORIAS`), `prioridade` (`baixa`, `media`, `alta`) e `resumo`. Use `"strict": True`, `additionalProperties: False` e todos os campos em `required`. Converta com `json.loads`.

### 2.9 **Módulo de saída estruturada.** Escreva `estruturado.py` com:
- a classe `Chamado` (pydantic) com `categoria: Literal[...]`, `prioridade: Literal["baixa", "media", "alta"]` e `resumo: str`;
- `classificar_chamado(texto)`: pede o JSON com o esquema gerado por `Chamado.model_json_schema()` (ajuste `additionalProperties`), valida com `Chamado.model_validate_json` e, se a validação falhar, tenta mais uma vez antes de desistir.

_A célula grava `meu_assistente/estruturado.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/estruturado.py


### 2.10 **Mede o classificador.** Rode `classificar_chamado` em todos os `TICKETS`, monte um `DataFrame` com o texto, a categoria certa e a prevista, e mostre a acurácia.

### 2.11 **O modelo é probabilístico.** Faça 5 vezes a mesma pergunta factual ("Em que ano foi lançado o primeiro iPhone? Responda só o ano.") com `temperature=1`, e depois 5 vezes com `temperature=0` e `seed=42`. Conte as respostas distintas. Qual configuração você usaria num produto?

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
